In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_curve,
    roc_auc_score,
    average_precision_score,
)
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
import numpy as np

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
import joblib
import pickle

# ── joblib 불러오기 ───────────────────────────
best_model = joblib.load("model/loan_default_model.joblib")
print(type(best_model))  # Pipeline인지 CatBoost 단독인지 확인

# ── pkl 불러오기 ──────────────────────────────
with open("model/catboost_model.pkl", "rb") as f:
    cat_model = pickle.load(f)
print(type(cat_model))

<class 'dict'>
<class 'catboost.core.CatBoostClassifier'>


In [8]:
# dict 키 확인
print(best_model.keys())

# 각 키별 타입 확인
for k, v in best_model.items():
    print(f"{k}: {type(v)}")

dict_keys(['model', 'threshold', 'features'])
model: <class 'catboost.core.CatBoostClassifier'>
threshold: <class 'float'>
features: <class 'list'>


In [4]:
use_col = ['loan_amnt', 'term', 'int_rate', 'installment', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'issue_d', 'purpose',
    'dti', 'earliest_cr_line', 'mths_since_last_delinq', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'mths_since_last_major_derog',
    'mths_since_rcnt_il', 'total_rev_hi_lim', 'acc_open_past_24mths',
    'avg_cur_bal', 'bc_open_to_buy', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mths_since_recent_bc',
       'mths_since_recent_bc_dlq', 'mths_since_recent_inq',
       'mths_since_recent_revol_delinq', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_il_tl', 'num_rev_tl_bal_gt_0', 'pct_tl_nvr_dlq', 'tot_hi_cred_lim',
       'total_bc_limit', 'issue_year', 'issue_month',
       'installment_to_income', 'loan_to_income', 'revol_bal_to_income',
       'fico_mid', 'mths_since_last_delinq_flag',
       'mths_since_last_major_derog_flag',
       'mths_since_recent_revol_delinq_flag', 'mths_since_recent_bc_dlq_flag',
       'mths_since_recent_inq_flag', 'mths_since_recent_bc_flag', 'target']

In [35]:
print(f"4번째 컬럼: {X_test_input.columns[3]}")
print(X_test_input.iloc[:, 3].dtype)
print(X_test_input.iloc[:, 3].head())

4번째 컬럼: installment
float64
211797    610.62
820519    284.30
974578    460.56
698374    622.04
596959    122.92
Name: installment, dtype: float64


In [6]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

use_col_with_target = use_col                                           # df2용 (target 포함)
use_col_features    = [col for col in use_col if col != "target"]      # X_test용 (target 제외)


# ── 데이터 불러오기 ───────────────────────────
df = pd.read_csv("lending_club_preprocessed3.csv")
df2 = df[use_col_with_target].copy()  # use_col 필터링

y = df2["target"]
X = df2.drop(columns=["target"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [7]:
# 파생변수 결측값 채우기
new_cols = ['installment_to_income', 'loan_to_income', 'revol_bal_to_income']
for col in new_cols:
    train_median = X_train[col].median()                    # train으로만 계산
    X_train[col] = X_train[col].fillna(train_median)       # train에 적용
    X_test[col] = X_test[col].fillna(train_median)         # test에도 train median 사용

# ★train/test 이후 전처리
# dti "sub_grade" 사용해야함 
X_train["dti"] = X_train.groupby("sub_grade")["dti"].transform(lambda x: x.fillna(x.median())
)
train_dit_medi = X_train.groupby("sub_grade")["dti"].median()
X_test["dti"] = X_test["dti"].fillna(X_test["sub_grade"].map(train_dit_medi))

# 신용 조회, 
cols = [
'mths_since_last_delinq',
'mths_since_last_major_derog',
'mths_since_recent_revol_delinq',
'mths_since_recent_bc_dlq',
'mths_since_recent_inq',
'mths_since_recent_bc',
'mths_since_rcnt_il',
'mo_sin_old_il_acct'
]
for col in cols:
    max_val = X_train[col].max()
    X_train[col] = X_train[col].fillna(max_val + 1)
    X_test[col] = X_test[col].fillna(max_val + 1)
    
# pct_tl_nvr_dlq 연체 이력이 한 번도 없는 계좌 비율 (%)
## 중앙값으로 채우기 
medi_dlq = X_train["pct_tl_nvr_dlq"].median()
X_train["pct_tl_nvr_dlq"] = X_train["pct_tl_nvr_dlq"].fillna(medi_dlq)
X_test["pct_tl_nvr_dlq"] = X_test["pct_tl_nvr_dlq"].fillna(medi_dlq)

In [17]:
# ── 모델 불러오기 ─────────────────────────────
best_model = joblib.load("model/loan_default_model2.joblib")
model     = best_model["model"]
threshold = best_model["threshold"]
features  = best_model["features"]

print(f"threshold: {threshold}")
print(f"features 수: {len(features)}")

# ── features 누락 확인 ────────────────────────
missing = [f for f in features if f not in X_test.columns]
print(f"누락 features: {missing}")  # 반드시 [] 나와야 함


# ── 예측 ──────────────────────────────────────
X_test_input = X_test[features]  # 모델에서 X값 
y_prob = model.predict_proba(X_test_input)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

# ── TP/TN/FP/FN 매핑 ──────────────────────────
test_df = X_test_input.copy()
test_df["target"] = y_test.values
test_df["y_pred"] = y_pred
test_df["y_prob"] = y_prob

def map_confusion(actual, pred):
    if actual == 1 and pred == 1:
        return "TP"
    elif actual == 0 and pred == 0:
        return "TN"
    elif actual == 0 and pred == 1:
        return "FP"
    else:
        return "FN"

test_df["confusion"] = test_df.apply(
    lambda row: map_confusion(row["target"], row["y_pred"]), axis=1
)

print(test_df["confusion"].value_counts())

threshold: 0.525
features 수: 39
누락 features: []
confusion
TN    143436
FP     60469
TP     32911
FN     18744
Name: count, dtype: int64


In [10]:
print("X_test_input 결측값 수:")
print(X_test_input.isnull().sum()[X_test_input.isnull().sum() > 0])

X_test_input 결측값 수:
Series([], dtype: int64)


In [13]:
# ── loan_amnt, int_rate, term 원본에서 가져오기 ─
test_df["loan_amnt"] = df.loc[test_df.index, "loan_amnt"]
test_df["int_rate"]  = df.loc[test_df.index, "int_rate"]
test_df["term"]      = df.loc[test_df.index, "term"]


# ── term 숫자로 변환 ──────────────────────────
test_df["term_num"] = test_df["term"].str.extract(r"(\d+)").astype(int)
test_df['rate'] = (test_df["int_rate"] / 100 / 12) # 월 이자율 파생컬럼

# ── interest 재정의 ───────────────────────────
test_df["interest"] = (
    (test_df["loan_amnt"] * (test_df["rate"]) * (1+test_df['rate'])**(test_df['term_num'])) / ((1+test_df['rate'])**(test_df['term_num']) - 1)
)
# installment = pv * rate * (1 + rate)**n / ((1 + rate)**n - 1)
             
  
# ────────────────────────────────────────────────
# 모델 미사용 손실 (A)
# 전부 승인 → 부도(TP+FN) 원금 손실
# ────────────────────────────────────────────────
tp_loss = test_df[test_df["confusion"] == "TP"]["loan_amnt"].sum()
fn_loss = test_df[test_df["confusion"] == "FN"]["loan_amnt"].sum()
A_loss  = tp_loss + fn_loss

# ────────────────────────────────────────────────
# 모델 사용 손실 (B)
# FN 원금 손실 + FP 이자 기회비용
# ────────────────────────────────────────────────
fp_opportunity = test_df[test_df["confusion"] == "FP"]["interest"].sum()
B_loss = fn_loss + fp_opportunity

# ── 손실 감소 효과 ────────────────────────────
loss_reduction     = A_loss - B_loss
loss_reduction_pct = (loss_reduction / A_loss) * 100

print("=" * 60)
print("📊 모델 사용 전후 손실 비교")
print("=" * 60)
print(f"[A] 모델 미사용 시 총 손실 (전부 승인)")
print(f"    ├ TP 원금 손실 (차단 가능했던):  -${tp_loss:>15,.2f}")
print(f"    ├ FN 원금 손실 (놓친 부도):      -${fn_loss:>15,.2f}")
print(f"    └ 총 손실:                       -${A_loss:>15,.2f}")
print(f"\n[B] 모델 사용 시 총 손실")
print(f"    ├ FN 원금 손실 (놓친 부도):      -${fn_loss:>15,.2f}")
print(f"    ├ FP 기회비용 (정상 거절):       -${fp_opportunity:>15,.2f}")
print(f"    └ 총 손실:                       -${B_loss:>15,.2f}")
print(f"\n손실 감소액 (A - B):                  ${loss_reduction:>15,.2f}")
print(f"손실 감소율:                          {loss_reduction_pct:>14.2f}%")
print("=" * 60)

# ── confusion별 상세 ──────────────────────────
print("\n📋 confusion별 상세")
detail = test_df.groupby("confusion").agg(
    건수=("loan_amnt", "count"),
    원금합계=("loan_amnt", "sum"),
    이자합계=("interest", "sum")
).round(2)
print(detail)

### 계산 구조

# A (미사용 손실) = TP 원금 + FN 원금
#                   (부도 전부 승인했을 때 손실)

# B (사용 손실)  = FN 원금 + FP 기회비용
#                   (모델 적용 후 남은 손실)

# 손실 감소액    = A - B  → TP가 차단한 원금 - FP 기회비용
# 손실 감소율    = (A - B) / A × 100

📊 모델 사용 전후 손실 비교
[A] 모델 미사용 시 총 손실 (전부 승인)
    ├ TP 원금 손실 (차단 가능했던):  -$ 557,565,450.00
    ├ FN 원금 손실 (놓친 부도):      -$ 254,222,250.00
    └ 총 손실:                       -$ 811,787,700.00

[B] 모델 사용 시 총 손실
    ├ FN 원금 손실 (놓친 부도):      -$ 254,222,250.00
    ├ FP 기회비용 (정상 거절):       -$  28,989,144.00
    └ 총 손실:                       -$ 283,211,394.00

손실 감소액 (A - B):                  $ 528,576,306.00
손실 감소율:                                   65.11%

📋 confusion별 상세
               건수          원금합계         이자합계
confusion                                   
FN          18744  2.542222e+08   7838079.65
FP          60469  9.768140e+08  28989144.00
TN         143436  1.923934e+09  59604996.74
TP          32911  5.575654e+08  16434296.95


In [12]:
# ── threshold별 손실 시뮬레이션 ──────────────
results = []

for thr in np.arange(0.1, 0.9, 0.05):
    y_pred_thr = (y_prob >= thr).astype(int)

    temp_df = test_df.copy()
    temp_df["y_pred"] = y_pred_thr
    temp_df["confusion"] = temp_df.apply(
        lambda row: map_confusion(row["target"], row["y_pred"]), axis=1
    )

    tp_l = temp_df[temp_df["confusion"] == "TP"]["loan_amnt"].sum()
    fn_l = temp_df[temp_df["confusion"] == "FN"]["loan_amnt"].sum()
    fp_o = temp_df[temp_df["confusion"] == "FP"]["interest"].sum()

    A = tp_l + fn_l
    B = fn_l + fp_o
    reduction = A - B

    counts = temp_df["confusion"].value_counts()
    results.append({
        "threshold":  round(thr, 2),
        "손실감소액": round(reduction, 2),
        "손실감소율": round((reduction / A) * 100, 2) if A > 0 else 0,
        "TN": counts.get("TN", 0),
        "FP": counts.get("FP", 0),
        "TP": counts.get("TP", 0),
        "FN": counts.get("FN", 0),
    })

result_df = pd.DataFrame(results)
print(result_df.to_string(index=False))

# ── 손실 감소액 최대 threshold ────────────────
best = result_df.loc[result_df["손실감소액"].idxmax()]
print(f"\n✅ 최적 threshold: {best['threshold']}")
print(f"   손실 감소액:     ${best['손실감소액']:,.2f}")
print(f"   손실 감소율:     {best['손실감소율']}%")
print(f"   TN: {int(best['TN']):,}  FP: {int(best['FP']):,}  TP: {int(best['TP']):,}  FN: {int(best['FN']):,}")

 threshold        손실감소액  손실감소율     TN     FP    TP    FN
      0.10 724597895.15  89.26   8966 194939 51501   154
      0.15 723158883.37  89.08  19667 184238 51087   568
      0.20 719392937.56  88.62  32606 171299 50374  1281
      0.25 710656882.72  87.54  47875 156030 49250  2405
      0.30 696972911.63  85.86  64470 139435 47658  3997
      0.35 675887505.67  83.26  82129 121776 45489  6166
      0.40 645093098.86  79.47 100183 103722 42607  9048
      0.45 606416278.23  74.70 118124  85781 39186 12469
      0.50 555864808.16  68.47 135295  68610 35076 16579
      0.55 497792368.69  61.32 151241  52664 30530 21125
      0.60 428665621.16  52.81 165657  38248 25562 26093
      0.65 353391790.54  43.53 177872  26033 20459 31196
      0.70 270555490.18  33.33 187649  16256 15259 36396
      0.75 185649838.17  22.87 195112   8793 10177 41478
      0.80 105475238.33  12.99 199981   3924  5667 45988
      0.85  43189169.75   5.32 202800   1105  2264 49391

✅ 최적 threshold: 0.1
   손실 감소액:

In [15]:
# ── loan_amnt, int_rate, term 원본에서 가져오기 ─
test_df["loan_amnt"] = df.loc[test_df.index, "loan_amnt"]
test_df["int_rate"]  = df.loc[test_df.index, "int_rate"]
test_df["term"]      = df.loc[test_df.index, "term"]

# ── term 숫자로 변환 ──────────────────────────
test_df["term_num"] = test_df["term"].str.extract(r"(\d+)").astype(int)
test_df["rate"]     = test_df["int_rate"] / 100 / 12  # 월 이자율

# ── 원리금 균등상환 방식 이자 계산 ───────────
# monthly_payment = PV * r * (1+r)^n / ((1+r)^n - 1)
# interest = total_payment - loan_amnt
test_df["monthly_payment"] = (
    test_df["loan_amnt"] * test_df["rate"] *
    (1 + test_df["rate"]) ** test_df["term_num"] /
    ((1 + test_df["rate"]) ** test_df["term_num"] - 1)
)
test_df["total_payment"] = test_df["monthly_payment"] * test_df["term_num"]
test_df["interest"]      = test_df["total_payment"] - test_df["loan_amnt"]

# ── confusion별 집계 ──────────────────────────
tn_interest = test_df[test_df["confusion"] == "TN"]["interest"].sum()
fp_interest = test_df[test_df["confusion"] == "FP"]["interest"].sum()
tp_loss     = test_df[test_df["confusion"] == "TP"]["loan_amnt"].sum()
fn_loss     = test_df[test_df["confusion"] == "FN"]["loan_amnt"].sum()

# ════════════════════════════════════════════════
# 모델 미사용 순이익 (A)
# = (TN + FP) 이자 수익 - (TP + FN) 원금 손실
# ════════════════════════════════════════════════
A_revenue = tn_interest + fp_interest
A_loss    = tp_loss + fn_loss
A_net     = A_revenue - A_loss

# ════════════════════════════════════════════════
# 모델 사용 순이익 (B)
# = TN 이자 + TP 막은 원금 - FN 원금 - FP 이자(기회비용)
# ════════════════════════════════════════════════
B_net = tn_interest + tp_loss - fn_loss - fp_interest

# ── 순이익 개선 효과 ──────────────────────────
net_improvement     = B_net - A_net
net_improvement_pct = (net_improvement / abs(A_net)) * 100

print("=" * 62)
print("📊 모델 사용 전후 순이익 비교 (원리금 균등상환 기준)")
print("=" * 62)
print(f"[A] 모델 미사용 순이익 (전부 승인)")
print(f"    ├ TN 이자 수익:              +${tn_interest:>15,.2f}")
print(f"    ├ FP 이자 수익:              +${fp_interest:>15,.2f}")
print(f"    ├ TP 원금 손실:              -${tp_loss:>15,.2f}")
print(f"    ├ FN 원금 손실:              -${fn_loss:>15,.2f}")
print(f"    └ 순이익:                     ${A_net:>15,.2f}")
print(f"\n[B] 모델 사용 순이익")
print(f"    ├ TN 이자 수익:              +${tn_interest:>15,.2f}")
print(f"    ├ TP 원금 방어 이득:         +${tp_loss:>15,.2f}")
print(f"    ├ FN 원금 손실:              -${fn_loss:>15,.2f}")
print(f"    ├ FP 이자 기회비용:          -${fp_interest:>15,.2f}")
print(f"    └ 순이익:                     ${B_net:>15,.2f}")
print(f"\n순이익 개선액 (B - A):            ${net_improvement:>15,.2f}")
print(f"순이익 개선율:                    {net_improvement_pct:>14.2f}%")
print("=" * 62)

# ── confusion별 상세 ──────────────────────────
print("\n📋 confusion별 상세 (원리금 균등상환 기준)")
detail = test_df.groupby("confusion").agg(
    건수=("loan_amnt", "count"),
    원금합계=("loan_amnt", "sum"),
    이자합계=("interest", "sum"),
    평균원금=("loan_amnt", "mean"),
    평균이자=("interest", "mean")
).round(2)
print(detail)

### 공식 구조 정리

# A_net = (TN이자 + FP이자) - (TP원금 + FN원금)
# B_net =  TN이자 + TP원금  -  FN원금 - FP이자

# B - A = TP원금×2 - FP이자×2
#       = 2 × (TP원금 - FP이자)

# → TP가 막은 원금 > FP 기회비용이면 B > A ✅
# → TP가 막은 원금 < FP 기회비용이면 B < A ❌

📊 모델 사용 전후 순이익 비교 (원리금 균등상환 기준)
[A] 모델 미사용 순이익 (전부 승인)
    ├ TN 이자 수익:              +$ 384,159,536.17
    ├ FP 이자 수익:              +$ 404,532,286.00
    ├ TP 원금 손실:              -$ 557,565,450.00
    ├ FN 원금 손실:              -$ 254,222,250.00
    └ 순이익:                     $ -23,095,877.83

[B] 모델 사용 순이익
    ├ TN 이자 수익:              +$ 384,159,536.17
    ├ TP 원금 방어 이득:         +$ 557,565,450.00
    ├ FN 원금 손실:              -$ 254,222,250.00
    ├ FP 이자 기회비용:          -$ 404,532,286.00
    └ 순이익:                     $ 282,970,450.18

순이익 개선액 (B - A):            $ 306,066,328.01
순이익 개선율:                           1325.20%

📋 confusion별 상세 (원리금 균등상환 기준)
               건수          원금합계          이자합계      평균원금     평균이자
confusion                                                       
FN          18744  2.542222e+08  5.939491e+07  13562.86  3168.74
FP          60469  9.768140e+08  4.045323e+08  16153.96  6689.91
TN         143436  1.923934e+09  3.841595e+08  13413.19  2678.26
TP          3291

In [16]:
# ── threshold별 순이익 시뮬레이션 ────────────
results = []

for thr in np.arange(0.1, 0.9, 0.05):
    y_pred_thr = (y_prob >= thr).astype(int)

    temp_df = test_df.copy()
    temp_df["y_pred"] = y_pred_thr
    temp_df["confusion"] = temp_df.apply(
        lambda row: map_confusion(row["target"], row["y_pred"]), axis=1
    )

    # ── confusion별 집계 ──────────────────────
    tn_i = temp_df[temp_df["confusion"] == "TN"]["interest"].sum()
    fp_i = temp_df[temp_df["confusion"] == "FP"]["interest"].sum()
    tp_l = temp_df[temp_df["confusion"] == "TP"]["loan_amnt"].sum()
    fn_l = temp_df[temp_df["confusion"] == "FN"]["loan_amnt"].sum()

    # ── 모델 미사용 순이익 (A) ────────────────
    # = (TN + FP) 이자 - (TP + FN) 원금
    A_net = (tn_i + fp_i) - (tp_l + fn_l)

    # ── 모델 사용 순이익 (B) ──────────────────
    # = TN 이자 + TP 원금 방어 - FN 원금 - FP 이자
    B_net = tn_i + tp_l - fn_l - fp_i

    # ── 순이익 개선액 ─────────────────────────
    improvement     = B_net - A_net
    improvement_pct = (improvement / abs(A_net)) * 100 if A_net != 0 else 0

    counts = temp_df["confusion"].value_counts()
    results.append({
        "threshold":    round(thr, 2),
        "A_순이익":     round(A_net, 2),
        "B_순이익":     round(B_net, 2),
        "순이익개선액": round(improvement, 2),
        "순이익개선율": round(improvement_pct, 2),
        "TN": counts.get("TN", 0),
        "FP": counts.get("FP", 0),
        "TP": counts.get("TP", 0),
        "FN": counts.get("FN", 0),
    })

result_df = pd.DataFrame(results)
print(result_df.to_string(index=False))

# ── 순이익 개선액 최대 threshold ──────────────
best = result_df.loc[result_df["순이익개선액"].idxmax()]
print(f"\n✅ 최적 threshold: {best['threshold']}")
print(f"   A 순이익 (미사용): ${best['A_순이익']:>15,.2f}")
print(f"   B 순이익 (사용):   ${best['B_순이익']:>15,.2f}")
print(f"   순이익 개선액:     ${best['순이익개선액']:>15,.2f}")
print(f"   순이익 개선율:     {best['순이익개선율']:>14.2f}%")
print(f"   TN: {int(best['TN']):,}  FP: {int(best['FP']):,}  TP: {int(best['TP']):,}  FN: {int(best['FN']):,}")

 threshold        A_순이익        B_순이익       순이익개선액  순이익개선율     TN     FP    TP    FN
      0.10 -23095877.83  43452546.48  66548424.31  288.14   8966 194939 51501   154
      0.15 -23095877.83  67232572.54  90328450.37  391.10  19667 184238 51087   568
      0.20 -23095877.83  97145318.79 120241196.62  520.62  32606 171299 50374  1281
      0.25 -23095877.83 130589175.16 153685052.99  665.42  47875 156030 49250  2405
      0.30 -23095877.83 166370113.05 189465990.88  820.35  64470 139435 47658  3997
      0.35 -23095877.83 201175313.79 224271191.62  971.04  82129 121776 45489  6166
      0.40 -23095877.83 230188919.40 253284797.23 1096.67 100183 103722 42607  9048
      0.45 -23095877.83 258248936.07 281344813.90 1218.16 118124  85781 39186 12469
      0.50 -23095877.83 274427126.97 297523004.80 1288.21 135295  68610 35076 16579
      0.55 -23095877.83 285110146.22 308206024.05 1334.46 151241  52664 30530 21125
      0.60 -23095877.83 280769123.16 303865000.99 1315.67 165657  38248 2556

In [ ]:
ssss